In [3]:
"""
BVEC Forecasting Model — Bayesian VECM with GLP Prior
======================================================
Real-time recursive forecasts of log real TTF NG prices.
Specifications: BVEC(1), BVEC(12), BVEC(AIC, p≤6).
Cointegrating rank r=1. Beta estimated by Johansen OLS at each origin.
GLP prior (Minnesota) applied to short-run parameters only.
Alpha (ECT coefficients) receives diffuse prior - not shrunk.
Lambda maximises multivariate log marginal likelihood at each origin.
"""

import numpy as np
import pandas as pd
from statsmodels.tsa.vector_ar.vecm import VECM
from scipy.optimize import minimize_scalar
import warnings
warnings.filterwarnings("ignore")

# ── Parameters ────────────────────────────────────────────────────────────────
HORIZONS     = [1, 3, 6, 9, 12, 15, 18, 21, 24]
EVAL_START   = "2015-01-01"
TRAIN_START  = "2006-02-01"
COINT_RANK   = 1
DET          = "ci"
AIC_LAG_MAX  = 6
INPUT_FILE   = "Input_VECM(Real_&_Log).xlsx"
OUTPUT_FILE  = "Output_BVEC_forecasts.xlsx"

VARS = ["log_ng", "log_oil", "log_coal"]

SPECIFICATIONS = [
    ("BVEC(1)",        1,    False),
    ("BVEC(12)",       12,   False),
    ("BVEC(AIC,p<=6)", None, True),
]

# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_excel(INPUT_FILE, sheet_name="Sheet1")
df.columns = ["date", "log_oil", "log_coal", "log_ng"]
df["date"]  = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
df = df[["date", "log_ng", "log_oil", "log_coal"]]

df_train = df[df["date"] >= TRAIN_START].reset_index(drop=True)

# ── Helper: actual real price for a given year-month ─────────────────────────
def get_actual(ym_str):
    m = df[df["date"].dt.to_period("M").astype(str) == ym_str]
    return np.exp(m["log_ng"].values[0]) if len(m) == 1 else np.nan

# ── Helper: build VECM regressor matrix ──────────────────────────────────────
def build_YX(data, p, beta, r=1):
    K   = data.shape[1]
    dy  = np.diff(data, axis=0)         # (T-1) x K
    n   = len(dy)
    ect = data[:-1] @ beta              # (T-1) x r

    cols = [ect]
    for lag in range(1, p):
        ld = np.zeros((n, K))
        for i in range(n):
            if i >= lag:
                ld[i] = dy[i - lag]
        cols.append(ld)
    cols.append(np.ones((n, 1)))

    X = np.hstack(cols)                 # (T-1) x n_params
    return dy, X                        # Y, X

# ── Helper: build prior variance diagonal ────────────────────────────────────
def build_V0_diag(lam, K, p, r, sigma2):
    v = []
    for k in range(K):
        # ECT coefficients: diffuse — preserve error correction mechanism
        for _ in range(r):
            v.append(1e6)
        # Lagged difference coefficients: Minnesota harmonic decay
        for lag in range(1, p):
            for j in range(K):
                if j == k:
                    v.append((lam / lag) ** 2)
                else:
                    v.append((lam / lag) ** 2 * sigma2[k] / sigma2[j])
        # Intercept: diffuse
        v.append(1e6)
    return np.array(v)

# ── Core: full multivariate GLP estimation ───────────────────────────────────
def bvec_glp(data, p, r=1, det="ci"):

#Estimate BVEC(p) using full multivariate GLP prior.

    K = data.shape[1]

    # Step 1: Johansen OLS to get beta
    vecm_ols = VECM(data, k_ar_diff=p-1, coint_rank=r,
                    deterministic=det).fit()
    beta = vecm_ols.beta                # K x r

    # Step 2: Build regressors
    Y, X  = build_YX(data, p, beta, r)
    n     = Y.shape[0]
    n_p   = X.shape[1]                 # params per equation

    # Step 3: OLS Sigma
    B_ols  = np.linalg.lstsq(X, Y, rcond=None)[0]
    E_ols  = Y - X @ B_ols
    Sigma  = E_ols.T @ E_ols / n
    sigma2 = np.diag(Sigma)
    Sig_inv = np.linalg.inv(Sigma)
    XtX    = X.T @ X

    # Step 4: Log marginal likelihood and optimisation
    def neg_lml(lam):
        if lam <= 1e-6:
            return np.inf
        V0d    = build_V0_diag(lam, K, p, r, sigma2)
        V0_inv = np.diag(1.0 / V0d)
        V1_inv = V0_inv + np.kron(Sig_inv, XtX)
        try:
            V1 = np.linalg.inv(V1_inv)
        except np.linalg.LinAlgError:
            return np.inf
        XtY_Sinv = X.T @ Y @ Sig_inv
        m1       = V1 @ XtY_Sinv.T.ravel()
        _, ld0   = np.linalg.slogdet(np.diag(V0d))
        _, ld1   = np.linalg.slogdet(V1)
        quad     = np.trace(Y.T @ Y @ Sig_inv) - m1 @ V1_inv @ m1
        return -(0.5 * ld1 - 0.5 * ld0 - 0.5 * quad)

    res     = minimize_scalar(neg_lml, bounds=(0.001, 100), method="bounded")
    lam_opt = res.x

    # Step 5: Posterior mean
    V0d    = build_V0_diag(lam_opt, K, p, r, sigma2)
    V0_inv = np.diag(1.0 / V0d)
    V1_inv = V0_inv + np.kron(Sig_inv, XtX)
    V1     = np.linalg.inv(V1_inv)
    XtY_Sinv = X.T @ Y @ Sig_inv
    m1     = V1 @ XtY_Sinv.T.ravel()
    B_post = m1.reshape(K, n_p).T      # n_params x K

    return B_post, beta, lam_opt

# ── Helper: iterated forecast ─────────────────────────────────────────────────
def bvec_forecast(B_post, beta, history, p, horizons, r=1):
    K      = history.shape[1]
    dy     = np.diff(history, axis=0)
    y_buf  = list(history)
    dy_buf = list(dy)
    fcsts  = {}

    for h in range(1, max(horizons) + 1):
        y_prev = np.array(y_buf[-1])
        ect    = beta.T @ y_prev        # r-vector

        x = [ect.flatten()]
        for lag in range(1, p):
            idx = -(lag)
            x.append(dy_buf[idx] if len(dy_buf) >= lag else np.zeros(K))
        x.append([1.0])
        x_vec  = np.concatenate(x)

        dy_hat = x_vec @ B_post         # K-vector
        y_hat  = y_prev + dy_hat

        y_buf.append(y_hat)
        dy_buf.append(dy_hat)

        if h in horizons:
            fcsts[h] = y_hat[0]         # log_ng

    return fcsts

# ── Helper: AIC lag selection ─────────────────────────────────────────────────
def select_aic_lag(data, max_lag, r=1, det="ci"):
    best_aic = np.inf
    best_p   = 1
    K        = data.shape[1]
    for p in range(1, max_lag + 1):
        k = p - 1
        try:
            res    = VECM(data, k_ar_diff=k, coint_rank=r,
                          deterministic=det).fit()
            n_p    = r + K * k + 1
            n_obs  = len(data) - 1
            aic    = -2 * res.llf + 2 * (K * n_p)
            if aic < best_aic:
                best_aic = aic
                best_p   = p
        except Exception:
            pass
    return best_p

# ── Main forecasting loop ─────────────────────────────────────────────────────
records      = []
eval_origins = df_train[df_train["date"] >= EVAL_START]["date"].tolist()

print(f"BVEC forecasting: {len(eval_origins)} origins x "
      f"{len(SPECIFICATIONS)} specs x {len(HORIZONS)} horizons")
print(f"Specifications:  {[s[0] for s in SPECIFICATIONS]}")
print(f"Coint rank:      r={COINT_RANK}  (imposed)")
print(f"Deterministic:   '{DET}'  (constant in coint relation, Case 3)")
print(f"Prior:           Full multivariate GLP (Giannone et al. 2015)")
print(f"AIC cap:         p<={AIC_LAG_MAX}  (Baumeister et al. 2024)")
print(f"Training:        {TRAIN_START} onwards")
print()

for i, origin_date in enumerate(eval_origins):

    history = df_train[df_train["date"] <= origin_date][VARS].values
    T       = len(history)

    if i % 20 == 0:
        print(f"  Origin {i+1}/{len(eval_origins)}: "
              f"{origin_date.strftime('%Y-%m-%d')}  T={T}")

    for label, fixed_p, use_aic in SPECIFICATIONS:

        p = select_aic_lag(history, AIC_LAG_MAX) if use_aic else fixed_p

        if T < p + 3:
            continue

        try:
            B_post, beta, lam_opt = bvec_glp(history, p)
            fcsts                 = bvec_forecast(B_post, beta, history, p, HORIZONS)
        except Exception:
            continue

        for h in HORIZONS:
            actual_ym = (origin_date + pd.DateOffset(months=h)).strftime("%Y-%m")
            records.append({
                "forecast_origin": origin_date.strftime("%Y-%m-%d"),
                "horizon":         h,
                "model":           label,
                "actual_month":    actual_ym,
                "forecast":        np.exp(fcsts[h]),
                "actual":          get_actual(actual_ym),
                "lag_order_used":  p,
                "lambda_opt":      round(lam_opt, 4),
            })

# ── Save output ───────────────────────────────────────────────────────────────
results = pd.DataFrame(records)
results.to_excel(OUTPUT_FILE, index=False)

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print("=" * 60)
print("BVEC FORECASTING COMPLETE")
print("=" * 60)
print(f"  Total rows:       {len(results)}")
print(f"  Forecast origins: {results['forecast_origin'].nunique()}")
print(f"  Output:           {OUTPUT_FILE}")
print()

for label, _, _ in SPECIFICATIONS:
    sub      = results[results["model"] == label]
    lag_dist = (sub.drop_duplicates("forecast_origin")["lag_order_used"]
                   .value_counts().sort_index().to_dict())
    lam_mean = sub.drop_duplicates("forecast_origin")["lambda_opt"].mean()
    lam_min  = sub.drop_duplicates("forecast_origin")["lambda_opt"].min()
    lam_max  = sub.drop_duplicates("forecast_origin")["lambda_opt"].max()
    print(f"  {label}:")
    print(f"    Origins:  {sub['forecast_origin'].nunique()}")
    print(f"    Lag dist: {lag_dist}")
    print(f"    Lambda:   mean={lam_mean:.3f}  min={lam_min:.3f}  max={lam_max:.3f}")
    print()

# VEC(AIC) lag distribution
aic_df = (results[results["model"] == "BVEC(AIC,p<=6)"]
          .drop_duplicates("forecast_origin").copy())
aic_df["forecast_origin"] = pd.to_datetime(aic_df["forecast_origin"])
aic_df = aic_df.sort_values("forecast_origin").reset_index(drop=True)
dist   = aic_df["lag_order_used"].value_counts().sort_index()
total  = len(aic_df)

print("BVEC(AIC) lag distribution:")
for p, count in dist.items():
    print(f"  p={p}: {count} origins ({count/total*100:.1f}%)")

# Sample — first origin
first_origin = results["forecast_origin"].min()
sample = results[results["forecast_origin"] == first_origin]
print(f"\nSample — first origin ({first_origin}):")
print(sample[["model","horizon","lag_order_used","lambda_opt",
              "actual_month","forecast","actual"]].to_string(index=False))

BVEC forecasting: 132 origins x 3 specs x 9 horizons
Specifications:  ['BVEC(1)', 'BVEC(12)', 'BVEC(AIC,p<=6)']
Coint rank:      r=1  (imposed)
Deterministic:   'ci'  (constant in coint relation, Case 3)
Prior:           Full multivariate GLP (Giannone et al. 2015)
AIC cap:         p<=6  (Baumeister et al. 2024)
Training:        2006-02-01 onwards

  Origin 1/132: 2015-01-31  T=108
  Origin 21/132: 2016-09-30  T=128
  Origin 41/132: 2018-05-31  T=148
  Origin 61/132: 2020-01-31  T=168
  Origin 81/132: 2021-09-30  T=188
  Origin 101/132: 2023-05-31  T=208
  Origin 121/132: 2025-01-31  T=228

BVEC FORECASTING COMPLETE
  Total rows:       3564
  Forecast origins: 132
  Output:           Output_BVEC_forecasts.xlsx

  BVEC(1):
    Origins:  132
    Lag dist: {1: 132}
    Lambda:   mean=100.000  min=100.000  max=100.000

  BVEC(12):
    Origins:  132
    Lag dist: {12: 132}
    Lambda:   mean=0.225  min=0.165  max=0.350

  BVEC(AIC,p<=6):
    Origins:  132
    Lag dist: {2: 84, 3: 48}
    La